# World Cup Predictor 2026 — Análise Exploratória (V1)

Notebook de demonstração do pipeline V1 com dados **mockados**.

Fluxo: histórico → Elo → forma recente → lambdas → Poisson → Monte Carlo → probabilidades.

> Os dados são fictícios. O objetivo é validar o framework e a calibração, não prever jogos reais.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_loader
from elo_model import build_elo_history
from feature_engineering import build_team_strengths, create_elo_diff
from goal_model import estimate_lambdas_for_fixture, calculate_score_matrix, probabilities_from_matrix
from monte_carlo import simulate_match

## 1. Histórico de partidas (mock)

In [ ]:
matches = data_loader.generate_mock_matches(n_matches=600)
print('Partidas:', len(matches))
print('Gols/time (média):', round(matches[['gols_time_a','gols_time_b']].values.mean(), 3))
matches.head()

## 2. Ratings Elo a partir do histórico

In [ ]:
ratings = build_elo_history(matches)
elo_df = pd.Series(ratings).sort_values(ascending=False)
elo_df.head(12).plot(kind='barh').invert_yaxis()
plt.title('Top seleções por Elo (dados mock)')
plt.xlabel('Elo')
plt.show()

## 3. Forças de ataque/defesa (forma recente)

In [ ]:
matches = create_elo_diff(matches, ratings)
strengths = build_team_strengths(matches, ratings, n_games=10)
strengths[['elo','forca_ofensiva','fragilidade_defensiva','saldo_medio_gols']].sort_values('elo', ascending=False).head(10)

## 4. Prever um confronto: lambdas + Poisson + Monte Carlo

In [ ]:
time_a, time_b = 'Brasil', 'Alemanha'
la, lb = estimate_lambdas_for_fixture(time_a, time_b, strengths,
                                      diferenca_elo=ratings[time_a]-ratings[time_b])
matrix = calculate_score_matrix(la, lb)
poisson_probs = probabilities_from_matrix(matrix)
mc = simulate_match(la, lb, n_simulations=10000, seed=1)

print(f'{time_a} x {time_b}')
print(f'lambda_a={la:.2f}  lambda_b={lb:.2f}')
print('Poisson :', {k: round(v,3) for k,v in poisson_probs.items() if k.startswith("prob_vitoria") or k=="prob_empate"})
print('MonteCarlo:', {k: round(v,3) for k,v in mc.items() if k.startswith("prob_vitoria") or k=="prob_empate"})

## 5. Matriz de placares

In [ ]:
plt.imshow(matrix, cmap='Blues', origin='lower')
plt.colorbar(label='probabilidade')
plt.xlabel(f'gols {time_b}')
plt.ylabel(f'gols {time_a}')
plt.title('Matriz de placares (Poisson)')
plt.show()

## 6. Calibração (backtest)

Veja `src/backtest.py` para o backtest walk-forward completo com Brier Score, Log Loss e curva de calibração.

```python
from backtest import run_backtest, _print_report
_print_report(run_backtest())
```